# Birdhaus Data Analysis

This notebook provides tools for exploring Birdhaus event, contact, and guest data.

## Setup

First, we load the data from the most recent CSV files.


In [1]:
import pandas as pd
import glob
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configure pandas display
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 50)


In [2]:
# Data loading utilities

def find_latest_csv(directory: Path, prefix: str) -> Path | None:
    """Find the most recently modified CSV file with the given prefix."""
    pattern = str(directory / f"{prefix}_*.csv")
    files = glob.glob(pattern)
    if not files:
        return None
    return Path(sorted(files, key=lambda x: Path(x).stat().st_mtime, reverse=True)[0])

# Find the project root (handles running from notebooks/ or project root)
notebook_dir = Path('.').resolve()
if notebook_dir.name == 'notebooks':
    PROJECT_ROOT = notebook_dir.parent
else:
    PROJECT_ROOT = notebook_dir

DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
VIEWS_DIR = PROJECT_ROOT / 'data' / 'views'

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Views directory: {VIEWS_DIR}")


Project root: /Users/saku/the-lab/birdhaus/birdhaus_data_pipeline
Data directory: /Users/saku/the-lab/birdhaus/birdhaus_data_pipeline/data/processed
Views directory: /Users/saku/the-lab/birdhaus/birdhaus_data_pipeline/data/views


In [3]:
# Load raw data
contacts_path = find_latest_csv(DATA_DIR, 'contacts')
events_path = find_latest_csv(DATA_DIR, 'events')
guests_path = find_latest_csv(DATA_DIR, 'guests')

print("Loading raw data:")
print(f"  Contacts: {contacts_path.name if contacts_path else 'NOT FOUND'}")
print(f"  Events: {events_path.name if events_path else 'NOT FOUND'}")
print(f"  Guests: {guests_path.name if guests_path else 'NOT FOUND'}")

contacts = pd.read_csv(contacts_path) if contacts_path else None
events = pd.read_csv(events_path) if events_path else None
guests = pd.read_csv(guests_path) if guests_path else None

print(f"\nLoaded:")
if contacts is not None: print(f"  {len(contacts):,} contacts")
if events is not None: print(f"  {len(events):,} events")
if guests is not None: print(f"  {len(guests):,} guests")


Loading raw data:
  Contacts: contacts_20251227_121228.csv
  Events: events_20251227_121226.csv
  Guests: guests_20251227_121233.csv

Loaded:
  1,639 contacts
  116 events
  3,829 guests


In [4]:
# Load enriched views (if available)
guests_enriched_path = find_latest_csv(VIEWS_DIR, 'guests_enriched')
contact_history_path = find_latest_csv(VIEWS_DIR, 'contact_event_history')
event_attendance_path = find_latest_csv(VIEWS_DIR, 'event_attendance')

print("Loading enriched views:")
print(f"  Guests Enriched: {guests_enriched_path.name if guests_enriched_path else 'NOT FOUND - run generate_views.py'}")
print(f"  Contact History: {contact_history_path.name if contact_history_path else 'NOT FOUND - run generate_views.py'}")
print(f"  Event Attendance: {event_attendance_path.name if event_attendance_path else 'NOT FOUND - run generate_views.py'}")

guests_enriched = pd.read_csv(guests_enriched_path) if guests_enriched_path else None
contact_history = pd.read_csv(contact_history_path) if contact_history_path else None
event_attendance = pd.read_csv(event_attendance_path) if event_attendance_path else None


Loading enriched views:
  Guests Enriched: guests_enriched_20251227_131111.csv
  Contact History: contact_event_history_20251227_131111.csv
  Event Attendance: event_attendance_20251227_131111.csv


---
## Data Overview

Let's explore the structure and content of each dataset.


In [5]:
# Quick overview of each dataset
print("=" * 60)
print("CONTACTS")
print("=" * 60)
if contacts is not None:
    print(f"Shape: {contacts.shape}")
    print(f"\nColumns: {list(contacts.columns)}")
    display(contacts.head(3))


CONTACTS
Shape: (1639, 36)

Columns: ['contact_id', 'revision', 'created_date', 'created_time', 'created_datetime', 'updated_date', 'updated_time', 'updated_datetime', 'primary_email', 'first_name', 'last_name', 'full_name', 'email', 'email_tag', 'email_count', 'picture_url', 'picture_width', 'picture_height', 'subscription_status', 'deliverability_status', 'is_member', 'member_id', 'member_status', 'member_email_verified', 'signup_date', 'signup_time', 'profile_nickname', 'profile_privacy', 'user_role', 'user_id', 'source_type', 'source_app_id', 'display_name_first_last', 'display_name_last_first', 'membership_status', 'is_mobile_member']


,contact_id,revision,created_date,created_time,created_datetime,updated_date,updated_time,updated_datetime,primary_email,first_name,last_name,full_name,email,email_tag,email_count,picture_url,picture_width,picture_height,subscription_status,deliverability_status,is_member,member_id,member_status,member_email_verified,signup_date,signup_time,profile_nickname,profile_privacy,user_role,user_id,source_type,source_app_id,display_name_first_last,display_name_last_first,membership_status,is_mobile_member
0,814125fd-c8b3-4be2-a28a-c36e13b23987,7,2024-10-22,13:38:19.863,2024-10-22T13:38:19.863Z,2025-11-07,20:00:05.459,2025-11-07T20:00:05.459Z,birdhausto@gmail.com,Birdhaus,Toronto,Birdhaus Toronto,birdhausto@gmail.com,UNTAGGED,1,https://lh3.googleusercontent.com/a/ACg8ocKm0N...,0.0,0.0,SUBSCRIBED,VALID,True,dca1b9ac-0700-4a92-8ded-359e209d186a,ACTIVE,True,2024-10-22,13:38:20,Birdhaus Studio,PUBLIC,OWNER,dca1b9ac-0700-4a92-8ded-359e209d186a,WIX_SITE_MEMBERS,eb377299-86b4-4a86-a1b5-774a2d1d374b,Birdhaus Toronto,Toronto Birdhaus,APPROVED,NaN
1,65cb5d79-4982-4550-96bd-e3b1fdf12a9d,7,2025-04-15,00:35:44.179,2025-04-15T00:35:44.179Z,2025-05-16,18:21:52.663,2025-05-16T18:21:52.663Z,adorned.ent@gmail.com,Natalie,Simon-Butler,Natalie Simon-Butler,adorned.ent@gmail.com,UNTAGGED,1,NaN,NaN,NaN,SUBSCRIBED,VALID,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,WIX_EVENTS,140603ad-af8d-84a5-2c80-a0f60cb47351,Natalie Simon-Butler,Simon-Butler Natalie,NaN,NaN
2,152a0000-c080-4bf1-b612-c9c9395b4223,7,2025-04-15,00:54:03.784,2025-04-15T00:54:03.784Z,2025-05-16,18:29:23.461,2025-05-16T18:29:23.461Z,gwenachalmers@gmail.com,Gwen,Chalmers,Gwen Chalmers,gwenachalmers@gmail.com,UNTAGGED,1,NaN,NaN,NaN,SUBSCRIBED,VALID,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,WIX_EVENTS,140603ad-af8d-84a5-2c80-a0f60cb47351,Gwen Chalmers,Chalmers Gwen,NaN,NaN


In [6]:
print("=" * 60)
print("EVENTS")
print("=" * 60)
if events is not None:
    print(f"Shape: {events.shape}")
    print(f"\nKey columns: event_id, title, status, start_date, primary_category")
    display(events[['event_id', 'title', 'status', 'start_date', 'primary_category']].head(5))


EVENTS
Shape: (116, 55)

Key columns: event_id, title, status, start_date, primary_category


,event_id,title,status,start_date,primary_category
0,ef7111c4-deb3-41b8-b50a-10c588dda8ff,"Your First Rope Class, Rope Singles Mixer and ...",UPCOMING,2026-01-09,Rope
1,762f84b5-f831-4aac-a2c7-fcb6febf0f6a,Non-Monogamy 201,UPCOMING,2026-01-23,Education and Culture
2,62d3e3f6-53d5-432c-a3a0-a861fc238c22,Bound Together,UPCOMING,2026-01-04,Rope
3,34efda7b-7316-41ac-802a-f218383599db,Smutty Storytelling,UPCOMING,2026-01-02,Education and Culture
4,c972cffe-a2df-4093-9f5c-68d2def2e2b7,Uncomfortable Transitions,UPCOMING,2026-01-25,NaN


In [7]:
print("=" * 60)
print("GUESTS")
print("=" * 60)
if guests is not None:
    print(f"Shape: {guests.shape}")
    print(f"\nKey columns: guest_id, event_id, contact_id, guest_type")
    display(guests[['guest_id', 'event_id', 'contact_id', 'guest_type', 'attendance_status']].head(5))


GUESTS
Shape: (3829, 31)

Key columns: guest_id, event_id, contact_id, guest_type


,guest_id,event_id,contact_id,guest_type,attendance_status
0,000360b7-3cce-4fc8-b643-1b1c5c221cb6,e56e5bd0-c3ae-4638-99bc-ffab2aaf9c9d,abb0bfd9-54f2-42bf-b406-1e322bda6c58,BUYER,ATTENDING
1,002003a6-638c-4fe5-86b2-3fbc6910d37c,2376cc04-6d0f-4d30-bc4f-7a0b67950fb8,c2fd93fb-3de9-4616-beed-82f573ca1e34,BUYER,ATTENDING
2,0028bcb0-86be-4222-8c5e-c0b1b760f64d,8d0d0298-4478-4fb9-ab79-19160b4e98e4,a25ee0e3-320a-402f-8078-bd4836ae303d,BUYER,ATTENDING
3,0058b840-9edd-4027-9af1-2deb570c543f,27249d3e-70b0-4fa9-996b-cd4874a95e3a,0f22056b-a6dd-49c4-b47a-b5bfdc4dc598,BUYER,ATTENDING
4,006f15e8-d666-4a72-ad95-c3d99679f951,da1808a3-d158-4a9d-bfd5-bf02918ad294,bba6baff-4b74-4fb6-8e07-6c55e9fb5bb5,BUYER,ATTENDING


---
## Analysis: Repeat Attendees

Who are the most frequent event attendees?


In [8]:
# Find repeat attendees using the contact_history view
if contact_history is not None:
    repeat_attendees = contact_history[
        contact_history['unique_events_count'] > 1
    ].sort_values('unique_events_count', ascending=False)
    
    print(f"Repeat attendees (attended more than 1 event): {len(repeat_attendees):,}")
    print(f"\nTop 20 most frequent attendees:")
    display(
        repeat_attendees[[
            'full_name', 'primary_email', 'unique_events_count', 
            'total_attendances', 'is_member'
        ]].head(20)
    )
else:
    print("Contact history view not available. Run generate_views.py first.")


Repeat attendees (attended more than 1 event): 287

Top 20 most frequent attendees:


,full_name,primary_email,unique_events_count,total_attendances,is_member
0,Stephen Akuffo,sa.akuffo@gmail.com,32,87,True
1,Amanda,amaccagnan@hotmail.com,25,69,False
2,Dominik,kopec.dominik@gmail.com,24,54,False
3,Ellen Moore,ellen.rebecca.moore@gmail.com,20,42,True
4,Alexander Ouellette,revers09.wispier@icloud.com,15,39,False
5,Vi Tourangeau,vincetogo@mac.com,15,33,False
7,Darwin LeMay,darweb2@yahoo.ca,13,28,False
6,Drew Petursson,peturssondrew@gmail.com,11,29,False
10,Rabinththan Ravichandran,rabinty99@gmail.com,10,20,True
18,Cacti C,runfromcacti@gmail.com,10,17,True


In [9]:
# Distribution of attendance counts
if contact_history is not None:
    attendance_dist = contact_history['unique_events_count'].value_counts().sort_index()
    print("Attendance distribution (events attended -> number of contacts):")
    print(attendance_dist.head(15))
    
    # Summary stats
    print(f"\nSummary:")
    print(f"  Contacts with 0 events: {(contact_history['unique_events_count'] == 0).sum():,}")
    print(f"  Contacts with 1 event: {(contact_history['unique_events_count'] == 1).sum():,}")
    print(f"  Contacts with 2+ events: {(contact_history['unique_events_count'] >= 2).sum():,}")
    print(f"  Contacts with 5+ events: {(contact_history['unique_events_count'] >= 5).sum():,}")
    print(f"  Contacts with 10+ events: {(contact_history['unique_events_count'] >= 10).sum():,}")


Attendance distribution (events attended -> number of contacts):
unique_events_count
0     672
1     680
2     168
3      46
4      30
5      13
6      12
7       4
8       2
9       1
10      3
11      1
13      1
15      2
20      1
Name: count, dtype: int64

Summary:
  Contacts with 0 events: 672
  Contacts with 1 event: 680
  Contacts with 2+ events: 287
  Contacts with 5+ events: 43
  Contacts with 10+ events: 11


---
## Analysis: Event Popularity

Which events have the most attendees?


In [10]:
# Most popular events by attendance
if event_attendance is not None:
    popular_events = event_attendance.sort_values('total_guests', ascending=False)
    
    print("Top 20 events by attendance:")
    display(
        popular_events[[
            'title', 'start_date', 'total_guests', 'unique_contacts',
            'buyer_count', 'ticket_holder_count', 'primary_category'
        ]].head(20)
    )
else:
    print("Event attendance view not available. Run generate_views.py first.")


Top 20 events by attendance:


,title,start_date,total_guests,unique_contacts,buyer_count,ticket_holder_count,primary_category
57,Sweat,2025-09-14,167,68,71,96,NaN
39,CONCLAVE - A Birdhaus Halloween,2025-11-02,164,64,64,100,Parties
62,OpenHaus,2025-09-06,159,68,68,91,Parties
97,Sweat,2025-06-01,157,58,65,92,NaN
69,Afternoon Delight,2025-08-23,150,64,64,86,Parties
80,Hentai Education and Nonsense,2025-07-20,131,53,56,75,Education and Culture
46,Voyeur,2025-10-12,126,45,46,80,Parties
15,Rocky Horror NYE!,2026-01-01,124,49,50,74,Parties
90,Voyeur,2025-06-14,123,40,43,80,Parties
40,Birdhaus Trick or Treat,2025-11-01,85,31,32,53,Parties


In [11]:
# Events with no attendees
if event_attendance is not None:
    no_attendees = event_attendance[event_attendance['total_guests'] == 0]
    print(f"Events with no registered guests: {len(no_attendees)}")
    if len(no_attendees) > 0:
        display(no_attendees[['title', 'start_date', 'status', 'primary_category']].head(10))


Events with no registered guests: 13


,title,start_date,status,primary_category
0,Uncomfortable Transitions,2026-01-25,UPCOMING,NaN
5,Taking to the Air,2026-01-18,UPCOMING,NaN
16,Self Tied Partials,2025-12-21,CANCELED,NaN
17,Futos and Frictions,2025-12-21,CANCELED,NaN
18,A Month of Mondays: Bedroom Bondage,2025-12-16,ENDED,Rope
22,A Month of Mondays: Bedroom Bondage,2025-12-09,ENDED,Rope
26,A Month of Mondays: Bedroom Bondage,2025-12-02,ENDED,Rope
30,Upright Suspension Poses for Impact and Sex,2025-11-30,CANCELED,Rope
32,Self Suspension; Taking to the Air,2025-11-23,ENDED,Rope
76,FaceUp/FaceDown - Exploring Single and Double ...,2025-07-27,ENDED,Rope


---
## Analysis: Event Categories

What types of events are most popular?


In [12]:
# Attendance by event category
if event_attendance is not None:
    category_stats = event_attendance.groupby('primary_category').agg({
        'event_id': 'count',
        'total_guests': 'sum',
        'unique_contacts': 'sum'
    }).rename(columns={
        'event_id': 'event_count',
        'total_guests': 'total_attendances',
        'unique_contacts': 'total_unique_contacts'
    })
    
    category_stats['avg_attendance'] = (
        category_stats['total_attendances'] / category_stats['event_count']
    ).round(1)
    
    category_stats = category_stats.sort_values('total_attendances', ascending=False)
    
    print("Attendance by event category:")
    display(category_stats)


Attendance by event category:


,event_count,total_attendances,total_unique_contacts,avg_attendance
primary_category,,,,
Parties,14,1267,497,90.5
Rope,57,1163,501,20.4
Education and Culture,14,532,224,38.0


---
## Analysis: Member vs Non-Member Attendance

How do members compare to non-members in event attendance?


In [13]:
# Member vs non-member comparison
if contact_history is not None:
    member_stats = contact_history.groupby('is_member').agg({
        'contact_id': 'count',
        'unique_events_count': ['sum', 'mean'],
        'total_attendances': ['sum', 'mean']
    }).round(2)
    
    member_stats.columns = [
        'contact_count', 'total_unique_events', 'avg_unique_events',
        'total_attendances', 'avg_attendances'
    ]
    
    print("Member vs Non-Member attendance:")
    display(member_stats)


Member vs Non-Member attendance:


,contact_count,total_unique_events,avg_unique_events,total_attendances,avg_attendances
is_member,,,,,
False,1471,1467,1.00,3414,2.32
True,168,182,1.08,415,2.47


---
## Analysis: Attendance Over Time

How has attendance changed over time?


In [14]:
# Attendance trend by month
if event_attendance is not None and 'start_date' in event_attendance.columns:
    ea = event_attendance.copy()
    ea['start_date'] = pd.to_datetime(ea['start_date'])
    ea['month'] = ea['start_date'].dt.to_period('M')
    
    # Only include past events
    past_events = ea[ea['start_date'] < datetime.now()]
    
    monthly_stats = past_events.groupby('month').agg({
        'event_id': 'count',
        'total_guests': 'sum',
        'unique_contacts': 'sum'
    }).rename(columns={
        'event_id': 'events_held',
        'total_guests': 'total_guests',
        'unique_contacts': 'unique_attendees'
    })
    
    print("Monthly attendance (past events only):")
    display(monthly_stats.tail(12))  # Last 12 months


Monthly attendance (past events only):


,events_held,total_guests,unique_attendees
month,,,
2025-04,5,54,24
2025-05,13,388,162
2025-06,12,439,166
2025-07,10,263,110
2025-08,13,343,154
2025-09,13,745,314
2025-10,9,382,152
2025-11,14,619,245
2025-12,11,117,51


---
## Custom Query Templates

Use these helper functions for your own exploratory queries.


In [15]:
# Find all events a specific contact attended
def find_contact_events(email_or_name: str):
    """Find all events attended by a contact (search by email or name)."""
    if guests_enriched is None:
        print("guests_enriched view not available")
        return None
    
    # Search by email or name
    mask = (
        guests_enriched['contact_primary_email'].str.contains(email_or_name, case=False, na=False) |
        guests_enriched['contact_full_name'].str.contains(email_or_name, case=False, na=False)
    )
    
    results = guests_enriched[mask][[
        'contact_full_name', 'contact_primary_email',
        'event_title', 'event_start_date', 'event_primary_category',
        'guest_type', 'order_number'
    ]].drop_duplicates()
    
    return results.sort_values('event_start_date', ascending=False)

# Try it:
# find_contact_events("example@email.com")


In [16]:
# Find all attendees of a specific event
def find_event_attendees(event_title_search: str):
    """Find all attendees of an event (search by title)."""
    if guests_enriched is None:
        print("guests_enriched view not available")
        return None
    
    mask = guests_enriched['event_title'].str.contains(event_title_search, case=False, na=False)
    
    results = guests_enriched[mask][[
        'event_title', 'event_start_date',
        'contact_full_name', 'contact_primary_email',
        'guest_type', 'attendance_status'
    ]].drop_duplicates()
    
    return results.sort_values(['event_title', 'contact_full_name'])

# Try it:
# find_event_attendees("Rope")


In [17]:
# Find contacts who attended events in a specific category
def find_category_attendees(category: str):
    """Find all unique contacts who attended events in a category."""
    if guests_enriched is None:
        print("guests_enriched view not available")
        return None
    
    # Check which column name exists for category
    cat_col = 'event_category_names' if 'event_category_names' in guests_enriched.columns else 'event_primary_category'
    
    mask = guests_enriched[cat_col].str.contains(category, case=False, na=False)
    
    results = guests_enriched[mask][[
        'contact_full_name', 'contact_primary_email', 'contact_is_member'
    ]].drop_duplicates()
    
    return results.sort_values('contact_full_name')

# Try it:
# find_category_attendees("Education")


---
## Your Custom Queries

Add your own exploration cells below:


In [18]:
# Your custom queries here:

